# BENZI — LoRA fine-tune on Colab (~2–3 hours)

**Runtime → Change runtime type → T4 GPU**

**If setup fails:** *Runtime → Disconnect and delete runtime*, then reopen this notebook from GitHub and Run all.

Open from: `github.com/sameedsaeed123/final-year-benzi` → `fyp-ml-demos/finetune/BENZI_Colab_Train.ipynb`


In [ ]:
# GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime → Change runtime type → GPU"
print("CUDA OK")


In [ ]:
# Setup repo at /content/final-year-benzi/fyp-ml-demos
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")
CONTENT = Path("/content")
REPO_DIR = CONTENT / "final-year-benzi"
ML_DIR = REPO_DIR / "fyp-ml-demos"
REPO_URL = "https://github.com/sameedsaeed123/final-year-benzi.git"

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("CMD:", " ".join(cmd))
        print(r.stderr or r.stdout)
    return r

# Must not be inside repo when deleting
if str(Path.cwd()).startswith(str(REPO_DIR)):
    os.chdir("/content")

if REPO_DIR.exists():
  run(["rm", "-rf", str(REPO_DIR)])
  if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR, ignore_errors=True)

if REPO_DIR.exists():
    raise RuntimeError(
        f"Cannot remove {REPO_DIR}. Use Runtime → Disconnect and delete runtime, then Run all."
    )

r = run([
    "git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR),
])
if r.returncode != 0:
    # Fallback: clone to temp dir then rename
    tmp = CONTENT / "final-year-benzi-tmp"
    run(["rm", "-rf", str(tmp)])
    r2 = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(tmp)],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        raise RuntimeError(
            "git clone failed. Check internet / GitHub access.\n" + (r2.stderr or r2.stdout)
        )
    tmp.rename(REPO_DIR)

os.chdir(ML_DIR)

train_py = ML_DIR / "finetune" / "train_qlora.py"
text = train_py.read_text(encoding="utf-8")
if "use_mps_device" in text:
    train_py.write_text(
        text.replace("        use_mps_device=False,\n", ""),
        encoding="utf-8",
    )
    print("Patched train_qlora.py")

assert train_py.is_file()
print("Working directory:", ML_DIR.resolve())


In [ ]:
# Install deps (Colab: keep numpy 2.x — do not use requirements-finetune.txt)
import subprocess, sys
from pathlib import Path

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-r", "requirements.txt")
colab = Path("requirements-finetune-colab.txt")
if colab.is_file():
    pip("-r", str(colab))
else:
    pip("datasets>=2.19.0", "peft>=0.11.0", "accelerate>=0.30.0",
        "sentencepiece>=0.2.0", "protobuf>=4.25.0", "tqdm>=4.66.0")
pip("bitsandbytes>=0.43.0", "accelerate")

import numpy as np, bitsandbytes as bnb, torch
print("numpy", np.__version__, "| bitsandbytes", bnb.__version__)


In [ ]:
# Step 1 — dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200


In [ ]:
# Step 2 — train (~45–90 min). Wait until you see: Saved LoRA adapter
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4


**Faster option (~20 min):** comment out Step 2 above and run:
```python
!python finetune/train_qlora.py --benzi-lite
```


In [ ]:
# Step 3 — merge (~10–25 min). Do NOT Ctrl+C at 0% — disk write is slow.
from pathlib import Path
import subprocess, sys

adapter_cfg = Path("finetune/adapters/benzi-lora/adapter_config.json")
if not adapter_cfg.is_file():
    raise RuntimeError("Run Step 2 first — adapter missing.")

# Uninstall broken torchao (upgrading causes wrong .so on Colab Python 3.12)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct


In [ ]:
# Step 4 — download zip
import shutil
from pathlib import Path
from google.colab import files

merged = Path("finetune/merged/benzi-empathetic-hf")
if not merged.is_dir():
    raise RuntimeError("Merged model missing — complete Step 3 first.")

shutil.make_archive("/content/benzi-empathetic-trained", "zip", merged)
files.download("/content/benzi-empathetic-trained.zip")
print("Download started: benzi-empathetic-trained.zip")


## On your Mac

```bash
mkdir -p ~/benzi-models && cd ~/benzi-models
unzip ~/Downloads/benzi-empathetic-trained.zip -d benzi-empathetic-hf
cd benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```

In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained` then restart the API.
